In [0]:
# One-time setup: install everything this notebook needs (Delta writes, ADLS
# access, and GTFS-realtime protobuf parsing), then restart Python so the
# newly installed packages are actually importable.
%pip install deltalake pandas pyarrow azure-storage-file-datalake azure-identity gtfs-realtime-bindings -q
dbutils.library.restartPython()


In [0]:
# Auth setup: builds the service-principal credential used everywhere below,
# plus ready-to-use clients for the "landing" and "bronze" containers.
# NOTE: restartPython() above wipes all Python state, so if you ever rerun
# Cell 1, you must rerun this cell too before anything else will work.
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

STORAGE_ACCOUNT = "adlsdataplatformstaging"

client_id     = dbutils.secrets.get("bootstrap-creds", "sp-client-id")
client_secret = dbutils.secrets.get("bootstrap-creds", "sp-client-secret")
tenant_id     = dbutils.secrets.get("bootstrap-creds", "sp-tenant-id")

credential = ClientSecretCredential(tenant_id, client_id, client_secret)

storage_options = {
    "account_name": STORAGE_ACCOUNT,
    "client_id": client_id,
    "client_secret": client_secret,
    "tenant_id": tenant_id,
}

service_client = DataLakeServiceClient(
    f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net", credential=credential
)
landing_client = service_client.get_file_system_client("landing")
bronze_client = service_client.get_file_system_client("bronze")


In [0]:
# Bronze ingestion, static schedule tables: reads the latest landed .txt file
# for each table, tags it with ingestion metadata, and appends to bronze as
# Delta. dtype=str keeps every column untyped at this layer (matches the real
# ingest_raw.py's inferSchema=false) and avoids the mixed-type Arrow crash we
# hit earlier on stop_times.
import pandas as pd
import io
from datetime import datetime, timezone
from deltalake import write_deltalake

TABLES = ["stops", "routes", "trips", "stop_times"]


def ingest_raw_txt(table: str):
    matches = [
        p.name for p in landing_client.get_paths(path="gtfs/schedule", recursive=True)
        if p.name.endswith(f"{table}.txt")
    ]
    if not matches:
        print(f"{table}: no files found in landing")
        return
    latest_path = sorted(matches)[-1]  # folder names sort chronologically (run_id/fetched_at)

    file_client = landing_client.get_file_client(latest_path)
    raw_bytes = file_client.download_file().readall()

    df = pd.read_csv(io.BytesIO(raw_bytes), dtype=str)
    df["_ingested_at"] = datetime.now(timezone.utc)
    df["_source_file"] = latest_path

    bronze_uri = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/gtfs/{table}/"
    write_deltalake(bronze_uri, df, mode="append", storage_options=storage_options)
    print(f"Wrote {len(df)} rows to {bronze_uri}")


for table in TABLES:
    ingest_raw_txt(table)


In [0]:
# Bronze ingestion, GTFS-realtime feeds: decodes the latest landed protobuf
# file per feed type into rows (parsing logic mirrors ingest_realtime.py
# exactly) and appends to bronze as Delta.
from google.transit import gtfs_realtime_pb2
from google.protobuf import json_format
import json

FEEDS = ["vehiclepos", "realtime", "alerts"]


def parse_vehiclepos(feed, source_file):
    rows = []
    for entity in feed.entity:
        v = entity.vehicle
        rows.append({
            "entity_id": entity.id,
            "trip_id": v.trip.trip_id,
            "route_id": v.trip.route_id,
            "vehicle_id": v.vehicle.id,
            "vehicle_label": v.vehicle.label,
            "latitude": v.position.latitude,
            "longitude": v.position.longitude,
            "bearing": v.position.bearing,
            "speed": v.position.speed,
            "current_stop_sequence": v.current_stop_sequence,
            "stop_id": v.stop_id,
            "vehicle_timestamp": v.timestamp,
            "_source_file": source_file,
        })
    return rows


def parse_realtime(feed, source_file):
    rows = []
    for entity in feed.entity:
        tu = entity.trip_update
        stop_time_updates = [json_format.MessageToDict(stu) for stu in tu.stop_time_update]
        rows.append({
            "entity_id": entity.id,
            "trip_id": tu.trip.trip_id,
            "route_id": tu.trip.route_id,
            "vehicle_id": tu.vehicle.id,
            "trip_timestamp": tu.timestamp,
            "delay": tu.delay,
            "stop_time_updates_json": json.dumps(stop_time_updates),
            "_source_file": source_file,
        })
    return rows


def parse_alerts(feed, source_file):
    rows = []
    for entity in feed.entity:
        a = entity.alert
        header = a.header_text.translation[0].text if a.header_text.translation else None
        description = a.description_text.translation[0].text if a.description_text.translation else None
        informed_entities = [json_format.MessageToDict(ie) for ie in a.informed_entity]
        rows.append({
            "entity_id": entity.id,
            "cause": a.cause,
            "effect": a.effect,
            "header_text": header,
            "description_text": description,
            "informed_entities_json": json.dumps(informed_entities),
            "_source_file": source_file,
        })
    return rows


FEED_PARSERS = {
    "vehiclepos": parse_vehiclepos,
    "realtime": parse_realtime,
    "alerts": parse_alerts,
}


def ingest_realtime_feed(feed_type: str):
    parse_fn = FEED_PARSERS[feed_type]

    matches = [
        p.name for p in landing_client.get_paths(path=f"gtfs/{feed_type}", recursive=True)
        if p.name.endswith("data.pb")
    ]
    if not matches:
        print(f"{feed_type}: no files found in landing")
        return
    latest_path = sorted(matches)[-1]

    file_client = landing_client.get_file_client(latest_path)
    raw_bytes = file_client.download_file().readall()

    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(raw_bytes)
    rows = parse_fn(feed, latest_path)

    if not rows:
        print(f"{feed_type}: feed decoded but had no entities")
        return

    df = pd.DataFrame(rows)
    df["_ingested_at"] = datetime.now(timezone.utc)

    bronze_uri = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/gtfs/{feed_type}/"
    write_deltalake(bronze_uri, df, mode="append", storage_options=storage_options)
    print(f"Wrote {len(df)} rows to {bronze_uri}")


for feed_type in FEEDS:
    ingest_realtime_feed(feed_type)


In [0]:
# Quick look at bronze: reads every table currently under gtfs/ in bronze
# (auto-discovered, not hardcoded) and previews row counts + a few rows each.
def read_bronze_table(table_path: str) -> pd.DataFrame:
    """Reads every Parquet file under a bronze Delta table folder into one
    pandas DataFrame, skipping Delta's _delta_log folder."""
    paths = bronze_client.get_paths(path=table_path, recursive=True)

    dfs = []
    for p in paths:
        if p.is_directory or "_delta_log" in p.name or not p.name.endswith(".parquet"):
            continue
        file_client = bronze_client.get_file_client(p.name)
        data = file_client.download_file().readall()
        dfs.append(pd.read_parquet(io.BytesIO(data)))

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


table_names = sorted({
    p.name.split("/")[1]
    for p in bronze_client.get_paths(path="gtfs", recursive=False)
    if p.is_directory
})

for table in table_names:
    df = read_bronze_table(f"gtfs/{table}")
    print(f"--- {table}: {len(df)} rows ---")
    display(df.head(5))


In [0]:
# Data quality rulebook: defines what "valid" means per table, plus two ways
# of applying those rules — build_valid_mask (row-level: which rows pass,
# used to split silver/quarantine) and validate_table (aggregate: did the
# table as a whole come out healthy, used just for reporting).
# Range/allowed-value checks coerce to numeric first, since bronze columns
# may be stored as strings (dtype=str) or as pandas-inferred numeric types
# depending on when/how each table was ingested.
DQ_RULES = {
    "stops": {
        "not_null": ["stop_id", "stop_name", "stop_lat", "stop_lon"],
        "unique": ["stop_id"],
        "ranges": {"stop_lat": (-90, 90), "stop_lon": (-180, 180)},
        "allowed_values": {
            "location_type": [0, 1, 2, 3, 4, 5, 6, 7, 8],
            "wheelchair_boarding": [0, 1, 2],
        },
    },
    "routes": {
        "not_null": ["route_id", "route_type"],
        "unique": ["route_id"],
        "allowed_values": {"route_type": [0, 1, 2, 3, 4, 5, 6, 7]},
    },
    "trips": {
        "not_null": ["route_id", "service_id", "trip_id"],
        "unique": ["trip_id"],
        "allowed_values": {
            "direction_id": [0, 1],
            "wheelchair_accessible": [0, 1, 2],
        },
    },
    "stop_times": {
        "not_null": ["trip_id", "stop_id", "stop_sequence", "arrival_time", "departure_time"],
        "unique": ["trip_id", "stop_sequence"],
        "allowed_values": {
            "pickup_type": [0, 1, 2, 3],
            "drop_off_type": [0, 1, 2, 3],
        },
    },
    "vehiclepos": {
        "not_null": ["entity_id", "trip_id", "vehicle_timestamp"],
        "unique": ["entity_id"],
        "ranges": {
            "latitude": (-90, 90),
            "longitude": (-180, 180),
            "bearing": (0, 360),
        },
    },
    "realtime": {
        "not_null": ["entity_id", "trip_id"],
        "unique": ["entity_id"],
    },
    "alerts": {
        "not_null": ["entity_id", "cause", "effect"],
        "unique": ["entity_id"],
        "allowed_values": {
            "cause": list(range(1, 13)),
            "effect": list(range(1, 12)),
        },
    },
}


def build_valid_mask(df: pd.DataFrame, rules: dict) -> pd.Series:
    mask = pd.Series(True, index=df.index)

    for column in rules.get("not_null", []):
        mask &= df[column].notna()

    for column, (min_value, max_value) in rules.get("ranges", {}).items():
        numeric_col = pd.to_numeric(df[column], errors="coerce")
        mask &= numeric_col.between(min_value, max_value)

    for column, allowed_values in rules.get("allowed_values", {}).items():
        numeric_col = pd.to_numeric(df[column], errors="coerce")
        mask &= numeric_col.isin(allowed_values)

    return mask


def check_not_null(df, columns):
    return {c: {"passed": df[c].isna().sum() == 0, "null_count": int(df[c].isna().sum())} for c in columns}

def check_unique(df, columns):
    dup = df.duplicated(subset=columns).sum()
    return {"columns": columns, "passed": dup == 0, "duplicate_count": int(dup)}

def check_range(df, column, min_value, max_value):
    numeric_col = pd.to_numeric(df[column], errors="coerce")
    invalid = (~numeric_col.between(min_value, max_value)).sum()
    return {"column": column, "passed": invalid == 0, "invalid_count": int(invalid)}

def check_allowed_values(df, column, allowed_values):
    numeric_col = pd.to_numeric(df[column], errors="coerce")
    invalid = (~numeric_col.isin(allowed_values)).sum()
    return {"column": column, "passed": invalid == 0, "invalid_count": int(invalid)}

def check_row_count(bronze_df, silver_df):
    return {"bronze_count": len(bronze_df), "silver_count": len(silver_df), "passed": len(silver_df) > 0}


def validate_table(table_name: str, silver_df: pd.DataFrame, bronze_df: pd.DataFrame = None) -> dict:
    rules = DQ_RULES[table_name]
    results = {}

    if "not_null" in rules:
        results["not_null"] = check_not_null(silver_df, rules["not_null"])
    if "unique" in rules:
        results["unique"] = check_unique(silver_df, rules["unique"])
    if "ranges" in rules:
        results["ranges"] = {c: check_range(silver_df, c, lo, hi) for c, (lo, hi) in rules["ranges"].items()}
    if "allowed_values" in rules:
        results["allowed_values"] = {c: check_allowed_values(silver_df, c, v) for c, v in rules["allowed_values"].items()}
    if bronze_df is not None:
        results["row_count"] = check_row_count(bronze_df, silver_df)

    return results


In [0]:
# Cleans one table: dedupes on its unique key (keeping the newest ingested
# copy), splits rows into valid/invalid using the rulebook above, and writes
# valid rows to silver + invalid rows to quarantine.
def clean_transform(table_name: str, bronze_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rules = DQ_RULES[table_name]

    dedup_keys = rules.get("unique")
    if dedup_keys:
        df = bronze_df.sort_values("_ingested_at").drop_duplicates(subset=dedup_keys, keep="last")
    else:
        df = bronze_df.drop_duplicates()

    mask = build_valid_mask(df, rules)
    valid_df, invalid_df = df[mask], df[~mask]

    silver_uri = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/gtfs/{table_name}/"
    write_deltalake(silver_uri, valid_df, mode="overwrite", schema_mode="overwrite", storage_options=storage_options)

    if len(invalid_df) > 0:
        quarantine_uri = f"abfss://quarantine@{STORAGE_ACCOUNT}.dfs.core.windows.net/gtfs/{table_name}/"
        write_deltalake(quarantine_uri, invalid_df, mode="append", storage_options=storage_options)
        print(f"{table_name}: {len(invalid_df)} rows quarantined")

    print(f"{table_name}: {len(valid_df)} valid rows written to silver")
    return valid_df, invalid_df


In [0]:
# Driver: runs clean_transform + validate_table for every table in DQ_RULES,
# printing a pass/fail summary for each and collecting results for later
# inspection (results_by_table["stop_times"], etc.).
results_by_table = {}

for table_name in DQ_RULES:
    print(f"=== {table_name} ===")
    try:
        bronze_df = read_bronze_table(f"gtfs/{table_name}")
        if bronze_df.empty:
            print(f"{table_name}: no data in bronze, skipping")
            continue

        valid_df, invalid_df = clean_transform(table_name, bronze_df)
        results = validate_table(table_name, silver_df=valid_df, bronze_df=bronze_df)
        results_by_table[table_name] = results
        print(results)

    except Exception as e:
        print(f"{table_name}: FAILED — {e}")

    print()


In [0]:
# Quick look at silver: same idea as the bronze peek cell, but reads whatever
# tables currently exist under gtfs/ in the silver container.
silver_client = service_client.get_file_system_client("silver")


def read_silver_table(table_path: str) -> pd.DataFrame:
    """Reads every Parquet file under a silver Delta table folder into one
    pandas DataFrame, skipping Delta's _delta_log folder."""
    paths = silver_client.get_paths(path=table_path, recursive=True)

    dfs = []
    for p in paths:
        if p.is_directory or "_delta_log" in p.name or not p.name.endswith(".parquet"):
            continue
        file_client = silver_client.get_file_client(p.name)
        data = file_client.download_file().readall()
        dfs.append(pd.read_parquet(io.BytesIO(data)))

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


silver_table_names = sorted({
    p.name.split("/")[1]
    for p in silver_client.get_paths(path="gtfs", recursive=False)
    if p.is_directory
})

for table in silver_table_names:
    df = read_silver_table(f"gtfs/{table}")
    print(f"--- {table}: {len(df)} rows ---")
    display(df.head(5))


In [0]:
# Full reset of landing, bronze, and silver (deliberately excludes quarantine,
# which wasn't part of the request) — wipes all data in these three
# containers so the pipeline can be rerun clean from scratch.
containers_to_wipe = ["landing", "bronze", "silver"]

for container in containers_to_wipe:
    fs_client = service_client.get_file_system_client(container)
    paths = list(fs_client.get_paths(recursive=False))

    if not paths:
        print(f"{container}: already empty")
        continue

    for p in paths:
        if p.is_directory:
            fs_client.get_directory_client(p.name).delete_directory()
        else:
            fs_client.get_file_client(p.name).delete_file()
        print(f"Deleted {container}/{p.name}")
